# Independent Project 3: Low-rank Models, Embeddings & Geometry
**100 points · 10 questions × 10**

Per question: **experiment executed & numbers reported (5)** · **explanation (5)**.
All formulas you need are printed in the Word handout.


In [ ]:
# ---- Setup (run once, Python 3.12) ----
%pip uninstall -y numpy scipy scikit-learn pandas matplotlib contourpy umap-learn

!rm -rf \
  /usr/local/lib/python3.12/dist-packages/numpy \
  /usr/local/lib/python3.12/dist-packages/numpy.libs \
  /usr/local/lib/python3.12/dist-packages/numpy-*.dist-info \
  /usr/local/lib/python3.12/dist-packages/scipy \
  /usr/local/lib/python3.12/dist-packages/scipy.libs \
  /usr/local/lib/python3.12/dist-packages/scipy-*.dist-info

%pip install -q --no-cache-dir \
  "numpy==1.26.4" \
  "scipy==1.15.3" \
  "scikit-learn==1.6.1" \
  "pandas==2.2.3" \
  "matplotlib==3.10.3" \
  "contourpy==1.3.2" \
  "umap-learn==0.5.7" \
  "nbformat==5.10.4" \
  "nbclient==0.10.2" \
  "ipykernel==6.29.5"

In [ ]:
import os, sys, time, warnings, importlib.metadata as meta
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.special import logsumexp
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.datasets import load_digits, make_swiss_roll, make_moons
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA, NMF, LatentDirichletAllocation, KernelPCA
from sklearn.manifold import Isomap, SpectralEmbedding, TSNE, trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
START = time.perf_counter()
plt.rcParams.update({'figure.dpi': 100, 'font.size': 9})
print('Python', sys.version.split()[0])
print({p: meta.version(p) for p in ['numpy', 'scipy', 'scikit-learn', 'pandas', 'matplotlib', 'umap-learn']})

def mse(a, b):
    return float(np.mean((a - b) ** 2))

def neighbors(X, k):
    D = pairwise_distances(X)
    np.fill_diagonal(D, np.inf)
    return np.argsort(D, axis=1, kind='stable')[:, :k]

def jaccard_reference(a, b):
    return float(np.mean([len(set(x) & set(y)) / len(set(x) | set(y)) for x, y in zip(a, b)]))

def quality(X, Y, k=10):
    return {'trustworthiness': trustworthiness(X, Y, n_neighbors=k), 'knn_Jaccard': jaccard_reference(neighbors(X, k), neighbors(Y, k))}

def nearest_word_labels(E, names, tol=1e-10):
    E = np.asarray(E, dtype=float)
    norms = np.linalg.norm(E, axis=1)
    valid = norms > 1e-12
    Z = np.divide(E, norms[:, None], out=np.zeros_like(E), where=norms[:, None] > 1e-12)
    S = Z @ Z.T
    S[:, ~valid] = -np.inf
    np.fill_diagonal(S, -np.inf)
    result = []
    for i in range(len(E)):
        if not valid[i] or not np.isfinite(S[i]).any():
            result.append('undefined: zero vector or no valid peer')
            continue
        top = np.max(S[i])
        tied = np.flatnonzero(np.isclose(S[i], top, rtol=0, atol=tol))
        label = str(names[tied[0]])
        if len(tied) > 1:
            label += f' [{len(tied)}-way tie]'
        result.append(label)
    return result
digits = load_digits()
rng = np.random.default_rng(10)
subset = rng.choice(len(digits.data), 600, replace=False)
D = digits.data[subset] / 16.0
labels = digits.target[subset]
train_idx, test_idx = train_test_split(np.arange(600), test_size=150, random_state=10, stratify=labels)
Xtr, Xte = (D[train_idx], D[test_idx])
assert Xtr.shape == (450, 64) and Xte.shape == (150, 64)
print('Offline datasets ready; digit train/test:', Xtr.shape, Xte.shape)


---
## Question 1 (10 pts):


In [ ]:
def pca_reconstruct(A, B, r):
    """Return reconstruction and retained training-variance fraction."""
    # TODO: implement using the Word handout.
    raise NotImplementedError

def pca_ref(A, B, r):
    model = PCA(n_components=r, svd_solver='full').fit(A)
    return (model.inverse_transform(model.transform(B)), model.explained_variance_ratio_.sum())

def run_pca(A, B, r):
    return pca_reconstruct(A, B, r)
A = np.array([[-1.0, -1.0], [1.0, 1.0]])
h, v = run_pca(A, A, 1)
assert np.allclose(h, A) and np.isclose(v, 1)
print('Q1 implementation checks PASSED')


In [ ]:
# Experiment code supplied.
q1 = []
recon = {}
for r in [2, 8, 16]:
    rec, var = run_pca(Xtr, Xte, r)
    ref, _ = pca_ref(Xtr, Xte, r)
    assert np.allclose(rec, ref, atol=1e-07)
    recon[r] = rec
    q1.append([r, var, mse(Xte, rec)])
q1 = pd.DataFrame(q1, columns=['rank', 'train_variance_fraction', 'test_MSE'])
display(q1)
fig, ax = plt.subplots(4, 5, figsize=(8, 6))
for row, (name, imgs) in enumerate([('Original', Xte)] + [(f'r={r}', recon[r]) for r in recon]):
    for j in range(5):
        ax[row, j].imshow(imgs[j].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
        ax[row, j].axis('off')
        ax[row, j].set_title(name if j == 0 else '')
plt.tight_layout()
plt.show()


**1. Compare at least two ranks using their retained-variance fractions and test MSEs. What information-cost tradeoff do the results support, and which rank would you select for reconstruction under a limited coordinate budget?**


**2. Does good reconstruction establish that digit classes are well separated? Explain what the reconstruction objective measures and give one additional evaluation you would need before recommending these coordinates for a classification or retrieval task.**




---
## Question 2 (10 pts):


In [ ]:
def nmf_h_update(X, W, H, eps=1e-12):
    # TODO: implement using the Word handout.
    return H * (W.T @ X) / (W.T @ W @ H + eps)
    raise NotImplementedError

def h_update(X, W, H):
    return nmf_h_update(X, W, H)
a = h_update(np.array([[6.0]]), np.array([[2.0]]), np.array([[1.0]]))
assert np.allclose(a, [[3.0]])
rng = np.random.default_rng(2)
xx = rng.random((6, 5))
ww = rng.random((6, 2))
hh = rng.random((2, 5))
new = h_update(xx, ww, hh)
assert new.shape == hh.shape and np.all(new >= 0)
assert np.linalg.norm(xx - ww @ new) <= np.linalg.norm(xx - ww @ hh) + 1e-08
print('Q2 implementation checks PASSED')


In [ ]:
# Experiment code supplied.
p8 = PCA(8, svd_solver='full').fit(Xtr)
q2 = [['PCA', 0, mse(Xte, p8.inverse_transform(p8.transform(Xte))), 0]]
bases = [('PCA signed bases', p8.components_)]
for seed in [0, 1]:
    model = NMF(n_components=8, init='random', random_state=seed, max_iter=1000, tol=0.001)
    model.fit(Xtr)
    rec = model.transform(Xte) @ model.components_
    q2.append(['NMF', seed, mse(Xte, rec), model.n_iter_])
    bases.append((f'NMF seed {seed}', model.components_))
display(pd.DataFrame(q2, columns=['method', 'seed', 'test_MSE', 'iterations']))
fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for i, (name, H) in enumerate(bases):
    for j in range(8):
        scale = np.abs(H).max()
        axes[i, j].imshow(H[j].reshape(8, 8), cmap='coolwarm' if i == 0 else 'gray', vmin=-scale if i == 0 else 0, vmax=scale)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(name)
plt.tight_layout()
plt.show()


**1.	Compare PCA and both NMF test errors and describe one visible difference in their bases. Does the lowest reconstruction error identify the most interpretable representation? Support your answer with both the numbers and basis images.**

**2.	Why need the two NMF runs not agree? Relate initialization and nonnegative additive components to their results, and explain why feeding ordinary NMF pixels standardized to negative values would violate its assumptions.**




---
## Question 3 (10 pts):


In [ ]:
def masked_rmse(R, pred, mask):
    # TODO: implement using the Word handout.
    return np.sqrt(np.mean((R[mask] - pred[mask] )** 2))
    raise NotImplementedError

def rmse_mask(R, P, M):
    assert M.any()
    return masked_rmse(R, P, M)
assert np.isclose(rmse_mask(np.array([5.0, 1.0, 99.0]), np.array([4.0, 3.0, 0.0]), np.array([True, True, False])), np.sqrt(2.5))
print('Q3 implementation checks PASSED')


In [ ]:
# Experiment code supplied.
rng = np.random.default_rng(3)
R = np.clip(3 + rng.normal(size=(60, 3)) @ rng.normal(size=(3, 40)) / 2 + rng.normal(0, 0.15, (60, 40)), 1, 5)
perm = rng.permutation(R.size)
tr = np.zeros(R.size, bool)
va = tr.copy()
te = tr.copy()
tr[perm[:1440]] = True
va[perm[1440:1920]] = True
te[perm[1920:]] = True
tr, va, te = [x.reshape(R.shape) for x in (tr, va, te)]
assert not np.any(tr & va | tr & te | va & te) and np.all(tr | va | te)
assert tr.sum(0).min() > 0 and tr.sum(1).min() > 0
print('Train / validation / test entries:', tr.sum(), va.sum(), te.sum())

def als(observed, mask, rank, lam=0.2, steps=20):
    rng = np.random.default_rng(0)
    U = rng.normal(0, 0.1, (observed.shape[0], rank))
    V = rng.normal(0, 0.1, (observed.shape[1], rank))
    for _ in range(steps):
        for i in range(len(U)):
            m = mask[i]
            B = V[m]
            U[i] = np.linalg.solve(B.T @ B + lam * np.eye(rank), B.T @ observed[i, m])
        for j in range(len(V)):
            m = mask[:, j]
            B = U[m]
            V[j] = np.linalg.solve(B.T @ B + lam * np.eye(rank), B.T @ observed[m, j])
    return U @ V.T
observed = np.where(tr, R, np.nan)
base = np.full_like(R, R[tr].mean())
rows = [['mean', rmse_mask(R, base, tr), rmse_mask(R, base, va)]]
val = {}
for r in [1, 3, 6]:
    pred = als(observed, tr, r)
    val[r] = rmse_mask(R, pred, va)
    rows.append([str(r), rmse_mask(R, pred, tr), val[r]])
q3 = pd.DataFrame(rows, columns=['rank/baseline', 'train_RMSE', 'validation_RMSE'])
display(q3)
best = min(val, key=val.get)
fit_mask = tr | va
final = als(np.where(fit_mask, R, np.nan), fit_mask, best)
final_mean = np.full_like(R, R[fit_mask].mean())
print('Validation-selected rank:', best)
print('Final test RMSE: selected ALS=', rmse_mask(R, final, te), 'mean=', rmse_mask(R, final_mean, te))


**1.	Which rank did validation select, and how did the refitted model compare with the mean baseline on test entries? Cite the relevant values and distinguish the role of validation from final evaluation.**

**2.	Why would zero-filled SVD solve a different problem? Discuss one way selectively missing ratings could undermine a conclusion based on this random-mask experiment.**


---
## Question 4 (10 pts):


In [ ]:
corpus = ['bread oven flour kitchen bake bread', 'soup kitchen vegetables cook soup', 'pasta tomato sauce kitchen cook', 'cake oven sugar flour bake', 'rice vegetables pan cook dinner', 'team goal football match team', 'coach team player football practice', 'tennis player court ball match', 'team stadium fans goal match', 'basketball coach court team ball', 'train ticket station travel train', 'hotel travel city room booking', 'flight airport ticket travel city', 'bus station ticket city trip', 'museum city hotel trip travel', 'team travel train ticket match', 'hotel kitchen cook pasta dinner', 'fans travel city stadium food kitchen']
cv = CountVectorizer()
C = cv.fit_transform(corpus).toarray().astype(float)
words = cv.get_feature_names_out()

def topic_word_prob(theta, phi):
    # TODO: implement using the Word handout.
    raise NotImplementedError

def topic_prob(theta, phi):
    return topic_word_prob(theta, phi)
assert np.allclose(topic_prob(np.array([[0.75, 0.25]]), np.array([[0.6, 0.4], [0.2, 0.8]])), [[0.5, 0.5]])

def plsa(C, K=3, steps=100):
    rng = np.random.default_rng(4)
    theta = rng.dirichlet(np.ones(K), len(C))
    phi = rng.dirichlet(np.ones(C.shape[1]), K)
    history = []
    for _ in range(steps):
        prob = topic_prob(theta, phi)
        history.append(float((C * np.log(np.maximum(prob, 1e-15))).sum()))
        resp = theta[:, :, None] * phi[None, :, :] / np.maximum(prob[:, None, :], 1e-15)
        expected = C[:, None, :] * resp
        theta = expected.sum(2)
        theta /= theta.sum(1, keepdims=True)
        phi = expected.sum(0)
        phi /= phi.sum(1, keepdims=True)
    return (theta, phi, np.array(history))
print('Q4 implementation checks PASSED')


In [ ]:
# Experiment code supplied.
theta, phi, hist = plsa(C)
assert np.allclose(theta.sum(1), 1) and np.allclose(phi.sum(1), 1)
assert np.min(np.diff(hist)) >= -1e-07
nm = NMF(3, init='nndsvda', random_state=0, max_iter=1000, tol=0.001)
W = nm.fit_transform(C)
models = {'pLSA': (theta, phi), 'NMF': (W / np.maximum(W.sum(1, keepdims=True), 1e-12), nm.components_)}
for alpha in [0.1, 1.0]:
    lda = LatentDirichletAllocation(3, doc_topic_prior=alpha, topic_word_prior=0.1, learning_method='batch', max_iter=40, random_state=0)
    mix = lda.fit_transform(C)
    models[f'LDA alpha={alpha}'] = (mix, lda.components_)
    entropy = -np.sum(mix * np.log(np.maximum(mix, 1e-15)), axis=1).mean()
    print('LDA alpha', alpha, 'mean entropy', entropy)
for name, (mix, basis) in models.items():
    print('\n', name)
    print({k: words[np.argsort(-row)[:5]].tolist() for k, row in enumerate(basis)})
    display(pd.DataFrame(mix, columns=['topic 0', 'topic 1', 'topic 2']).round(3))
plt.plot(hist)
plt.xlabel('EM iteration')
plt.ylabel('training log-likelihood')
plt.title('pLSA numerical check')
plt.show()


**1.	Name the learned topics from their highest-weight words and compare one mixed document across methods. Explain why topic indices need not align and why normalized NMF weights are not automatically posterior probabilities.**

**2.	Compare the two LDA mixture entropies. How does the observed result relate to the intended effect of alpha? Explain why neither a coherent topic name nor high in-sample likelihood alone establishes generalization.**



---
## Question 5 (10 pts):


In [ ]:
word_names = np.array(['coffee', 'tea', 'bread', 'train', 'bus', 'ticket'])
contexts = ['cup', 'drink', 'breakfast', 'station', 'ride', 'fare']
co = np.array([[40, 30, 15, 1, 0, 1], [30, 40, 10, 0, 1, 0], [10, 5, 40, 0, 0, 1], [0, 1, 0, 40, 30, 20], [0, 0, 1, 30, 40, 20], [1, 0, 0, 20, 20, 40]], dtype=float)
display(pd.DataFrame(co, index=word_names, columns=contexts))

def ppmi(C):
    """Return a finite matrix with the same shape as C."""
    # TODO: implement using the Word handout.
    raise NotImplementedError

def run_ppmi(C):
    return ppmi(C)
tiny = np.array([[3.0, 1.0], [1.0, 3.0]])
assert np.allclose(run_ppmi(tiny), np.diag([np.log(1.5)] * 2))
assert np.allclose(run_ppmi(np.array([[1.0, 0.0], [0.0, 1.0]])), np.diag([np.log(2)] * 2))
assert np.isfinite(run_ppmi(np.array([[1.0, 0.0], [0.0, 0.0]]))).all()
print('Q5 implementation checks PASSED')


In [ ]:
# Experiment code supplied.
M = run_ppmi(co)
assert M.shape == co.shape and np.isfinite(M).all() and np.all(M >= 0)
assert np.all(M[co == 0] == 0)
display(pd.DataFrame(M, index=word_names, columns=contexts))
pair_rows = []
for word, context in [('coffee', 'cup'), ('ticket', 'fare'), ('train', 'ride')]:
    i = np.flatnonzero(word_names == word)[0]
    j = contexts.index(context)
    expected = co[i].sum() * co[:, j].sum() / co.sum()
    pair_rows.append([word, context, co[i, j], expected, co[i, j] / expected, np.log(co[i, j] / expected)])
q5_pairs = pd.DataFrame(pair_rows, columns=['word', 'context', 'observed', 'expected', 'observed_over_expected', 'PMI'])
display(q5_pairs)
U, s, Vt = np.linalg.svd(M, full_matrices=False)
q5 = []
q5_neighbors = pd.DataFrame({'word': word_names, 'count_cosine': nearest_word_labels(co, word_names), 'PPMI_cosine': nearest_word_labels(M, word_names)})
for rank in [1, 2, 3]:
    E = U[:, :rank] * np.sqrt(s[:rank])
    rec = U[:, :rank] * s[:rank] @ Vt[:rank]
    q5.append([rank, mse(M, rec), int(np.sum(np.linalg.norm(E, axis=1) <= 1e-12))])
    q5_neighbors[f'SVD_rank_{rank}'] = nearest_word_labels(E, word_names)
q5 = pd.DataFrame(q5, columns=['rank', 'PPMI_reconstruction_MSE', 'zero_vector_count'])
display(q5)
display(q5_neighbors)


**1.	Use two audited word-context pairs to compare raw counts with their observed/expected ratios and PMI. Which pair is more strongly associated relative to its baseline, and why need that ranking not follow raw frequency?**

**2.	Use the reconstruction and nearest-word results to assess the effect of SVD rank, including ties or zero vectors. Explain why shared context profiles can support similar word vectors even without direct word-word co-occurrence, and why association with a context is not proof of synonymy.**


---
## Question 6 (10 pts):


In [ ]:
sentence_roles = pd.DataFrame([['Anchor', 'The customer wants a refund.'], ['Positive', 'The buyer is asking for their money back.'], ['Easy negative', 'The train arrives at noon.'], ['Difficult negative', 'The customer does not want a refund.']], columns=['role', 'sentence'])
display(sentence_roles)

def contrastive_loss(similarities, tau):
    """Return one scalar loss; the first candidate is positive."""
    # TODO: implement using the Word handout.
    raise NotImplementedError

def nce(s, tau):
    if tau <= 0:
        raise ValueError('temperature must be positive')
    s = np.asarray(s, dtype=float)
    return contrastive_loss(s, tau)
assert np.isclose(nce([1, 0], 1), np.log1p(np.exp(-1)))
assert np.isclose(nce([1, 1], 1), np.log(2))
assert np.isfinite(nce([1000, 0], 0.1))
assert np.isclose(nce([0.8, 0.1, 0.2], 0.2), nce([10.8, 10.1, 10.2], 0.2))
print('Q6 implementation checks PASSED')


In [ ]:
# Experiment code supplied.
rows = []
for tau in [0.2, 1.0]:
    for kind, sims in [('easy', [0.8, 0.1, 0.2]), ('hard', [0.8, 0.1, 0.9])]:
        loss = nce(sims, tau)
        rows.append([tau, kind, loss, np.exp(-loss)])
q6_loss = pd.DataFrame(rows, columns=['temperature', 'negative_set', 'loss', 'positive_share'])
display(q6_loss)


**1.	Use all four losses and positive shares to explain the effect of the harder competitor and temperature. Why would a lower loss obtained solely by changing temperature not establish that the encoder improved?**

**2.	Propose a positive-pair rule for request-intent matching and give a pair that shares the refund topic but has different intent or state. Explain how a genuinely matching request used as a negative could conflict with that task.**


---
## Question 7 (10 pts):


In [ ]:
Xroll, troll = make_swiss_roll(n_samples=450, noise=0, random_state=7)

def two_nn(X):
    """Return a scalar intrinsic-dimension estimate."""
    # TODO: implement using the Word handout.
    raise NotImplementedError

def id_est(X):
    return two_nn(X)
check = np.random.default_rng(77).normal(size=(80, 3))
assert np.isclose(id_est(check), id_est(3 * check + 10), rtol=1e-07)
try:
    id_est(np.array([[0.0, 0.0], [0.0, 0.0], [1.0, 0.0]]))
except ValueError:
    pass
else:
    raise AssertionError('Reject duplicate points explicitly')
print('Q7 implementation checks PASSED')


In [ ]:
# Experiment code supplied.
noise = np.random.default_rng(7).normal(size=Xroll.shape)
q7_id = pd.DataFrame([[sigma, id_est(Xroll + sigma * noise)] for sigma in [0, 0.2, 0.8]], columns=['noise_sd', 'TwoNN_dimension'])
display(q7_id)
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(projection='3d')
ax.scatter(*Xroll.T, c=troll, s=8, cmap='viridis')
ax.set_title('Swiss roll: t used only as color')
plt.show()
q7 = []
graph_embeddings = {}
for k in [6, 12, 24]:
    idx = neighbors(Xroll, k)
    G = np.zeros((len(Xroll), len(Xroll)))
    G[np.arange(len(Xroll))[:, None], idx] = 1
    G = np.maximum(G, G.T)
    components = connected_components(csr_matrix(G), directed=False, return_labels=False)
    print('Graph k=', k, 'connected components=', components)
    if components != 1:
        q7.append({'graph_k': k, 'components': components, 'method': 'SKIPPED disconnected'})
        continue
    dist = pairwise_distances(Xroll)
    geo_input = np.where(G > 0, dist, 0)
    iso = Isomap(n_neighbors=None, radius=np.inf, n_components=2, metric='precomputed', eigen_solver='arpack')
    Yi = iso.fit_transform(csr_matrix(geo_input))
    Ys = SpectralEmbedding(2, affinity='precomputed', random_state=0).fit_transform(G)
    for name, Y in [('Isomap', Yi), ('Laplacian', Ys)]:
        graph_embeddings[name, k] = Y
        q7.append(dict(graph_k=k, components=components, method=name, **quality(Xroll, Y)))
    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    for a, (name, Y) in zip(ax, [('Isomap', Yi), ('Laplacian', Ys)]):
        a.scatter(Y[:, 0], Y[:, 1], c=troll, s=8, cmap='viridis')
        a.set_title(f'{name}, graph k={k}')
    plt.tight_layout()
    plt.show()
q7 = pd.DataFrame(q7)
display(q7)


**1.	Compare ambient dimension, estimated intrinsic dimension and display dimension using your three noise-dependent estimates. What does the noise experiment suggest, and why should the estimate not be treated as an exact integer or proof of a particular manifold?**

**2.	Cite two graph/quality results to discuss neighborhood scale, connectivity and possible shortcuts across folds. Why does a good local-neighbor score against ambient 3D distances not by itself establish preservation of global surface geodesics?**


---
## Question 8 (10 pts):


In [ ]:
def mean_jaccard(a, b):
    # TODO: implement using the Word handout.
    raise NotImplementedError

def overlap(a, b):
    return mean_jaccard(a, b)
assert np.isclose(overlap(np.array([[1, 2, 3]]), np.array([[2, 3, 4]])), 0.5)
assert np.isclose(overlap(np.array([[1, 2]]), np.array([[1, 2]])), 1)
assert np.isclose(overlap(np.array([[1, 2]]), np.array([[3, 4]])), 0)
print('Q8 implementation checks PASSED')


In [ ]:
# Experiment code supplied.
import umap
q8 = []
maps = {}
for method in ['t-SNE', 'UMAP']:
    fig, ax = plt.subplots(2, 2, figsize=(8, 6))
    for row, param in enumerate([10, 40]):
        for seed in [0, 1]:
            if method == 't-SNE':
                Y = TSNE(n_components=2, perplexity=param, random_state=seed, init='random', learning_rate='auto', max_iter=600).fit_transform(Xroll)
            else:
                Y = umap.UMAP(n_components=2, n_neighbors=param, min_dist=0.1, random_state=seed, n_jobs=1).fit_transform(Xroll)
            maps[method, param, seed] = Y
            q8.append([method, param, seed, trustworthiness(Xroll, Y, n_neighbors=10), overlap(neighbors(Xroll, 10), neighbors(Y, 10))])
            ax[row, seed].scatter(Y[:, 0], Y[:, 1], c=troll, s=8, cmap='viridis')
            ax[row, seed].set_title(f'{method}: parameter={param}, seed={seed}')
    plt.tight_layout()
    plt.show()
q8 = pd.DataFrame(q8, columns=['method', 'perplexity_or_n_neighbors', 'seed', 'trustworthiness_k10', 'Jaccard_k10'])
display(q8)


**1.	Cite one within-method parameter comparison and one within-method seed comparison using both metrics. Explain why equal Jaccard overlap can hide different intrusion ranks, and which result appears more stable at the chosen evaluation scale.**

**2.	Identify a visual claim about island area, density or between-island distance that the plots do not justify. Explain how you would validate a clustering pipeline that uses a reduced representation instead of choosing it from visual separation alone.**


---
## Question 9 (10 pts):


In [ ]:
def rbf_kernel(X, gamma):
    """Return the n-by-n kernel matrix."""
    # TODO: implement using the Word handout.
    raise NotImplementedError

def kernel(X, gamma):
    return rbf_kernel(X, gamma)
assert np.allclose(kernel(np.array([[0.0], [1.0]]), 2), [[1, np.exp(-2)], [np.exp(-2), 1]])
print('Q9 implementation checks PASSED')


In [ ]:
# Experiment code supplied.
Xm, ym = make_moons(n_samples=240, noise=0.06, random_state=9)
q9_kernel = []
fig, ax = plt.subplots(1, 4, figsize=(13, 3))
ax[0].scatter(*Xm.T, c=ym, s=10)
ax[0].set_title('Input moons')
for a, gamma in zip(ax[1:], [0.2, 2.0, 20.0]):
    K = kernel(Xm, gamma)
    assert K.shape == (len(Xm), len(Xm)) and np.isfinite(K).all()
    assert np.allclose(K, K.T) and np.allclose(np.diag(K), 1)
    assert np.linalg.eigvalsh(K).min() > -1e-08
    Y = KernelPCA(n_components=2, kernel='precomputed', eigen_solver='arpack', random_state=0).fit_transform(K)
    q9_kernel.append(dict(gamma=gamma, mean_off_diagonal_kernel=K[~np.eye(len(K), dtype=bool)].mean(), **quality(Xm, Y)))
    a.scatter(*Y.T, c=ym, s=10)
    a.set_title(f'Kernel PCA, gamma={gamma}')
plt.tight_layout()
plt.show()
q9_kernel = pd.DataFrame(q9_kernel)
display(q9_kernel)
linear = PCA(8, svd_solver='full').fit(Xtr)
lintr = linear.inverse_transform(linear.transform(Xtr))
linte = linear.inverse_transform(linear.transform(Xte))
ae = MLPRegressor(hidden_layer_sizes=(8,), activation='tanh', solver='adam', alpha=0.0001, batch_size=64, learning_rate_init=0.005, max_iter=250, tol=1e-05, n_iter_no_change=25, random_state=0)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    ae.fit(Xtr, Xtr)
ae_warnings = [str(w.message) for w in caught]
print('AE iterations:', ae.n_iter_, '; convergence warnings:', ae_warnings)
q9_ae = pd.DataFrame([['linear PCA/AE reference', mse(Xtr, lintr), mse(Xte, linte)], ['nonlinear AE', mse(Xtr, ae.predict(Xtr)), mse(Xte, ae.predict(Xte))]], columns=['model', 'train_MSE', 'test_MSE'])
display(q9_ae)


**1.	Use kernel-similarity and neighborhood results from at least two gamma values to explain how the similarity scale changed. Why do these full-dataset pictures and local metrics not establish a uniquely correct representation or held-out generalization?**

**2.	Compare the two autoencoder/PCA test errors, citing iteration or warning information where relevant. Must finite nonlinear training beat PCA? Explain how identical reconstruction can coexist with different latent distances and why reconstruction alone does not validate a search space.**


---
## Question 10 (10 pts):


**1.	In 150-200 words, respond to the reviewer using at least four numerical results with settings, including completion validation/test results and a neighborhood audit. Distinguish intrinsic, representation and display dimensions, and explain why matching dimensions does not make the objectives equivalent.**

**2.	In 100-150 words, propose one controlled follow-up experiment. State the task, what stays fixed, the factor changed, metric and predicted result. Give a concrete outcome that would change your recommendation and one limitation the experiment would leave unresolved.**
